In [ ]:
#FORWARD CHAINING ALGORITHM

#import tools
import csv #reading required file
import random #to shuffle cars before splitting into training and test data
from collections import Counter #counts (like cars per class)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix #scoring tools 

#no header row in car.data so columns named manually - rows become dictionaries
columns = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
with open("car.data") as f:
    rows = [dict(zip(columns, line)) for line in csv.reader(f) if line] #if line skips blanks

random.seed(71)#fixed seed so same split each run, results thereby reproducible. shuffling both groups gets a mix of classes
shuffled = rows[:] #copy so OG order remains
random.shuffle(shuffled) #mixy mix cars groups to get fair combination.else test set would only have a certain group of cars since car.data is sorted

cut = int(0.7 * len(shuffled)) #70% of 1728, 1209 = training data to design rules. the 519 test cars used to score the rules
train, test = shuffled[:cut], shuffled[cut:] #train = design rues. test = score rules. 70% train, 30% test

#rules:
    #set of conditions, conclusion and fires when all conditions = known facts. 
rules = [
#non-negotiable rules first
    ({"safety=low"}, "class=unacc"), 
    ({"persons=2"}, "class=unacc"),
    ({"buying=vhigh", "maint=vhigh"}, "class=unacc"),
    ({"buying=vhigh", "maint=high"}, "class=unacc"),
    ({"buying=high", "maint=vhigh"}, "class=unacc"),

    ({"buying=low", "safety=high", "lug_boot=big"}, "class=vgood"), 
    ({"buying=med", "safety=high", "lug_boot=big"}, "class=vgood"), 

    ({"buying=low", "maint=low"}, "class=good"), 
    ({"safety=high"}, "class=acc"),
    ({"safety=med", "lug_boot=big"}, "class=acc")

] # no rule fires meaning the default response is class=unacc which is set in a later predict function

#start from known facts. fire until conditions known + conclusion added as new fact
def forward_chain(facts, rules):
    known = set(facts) #a copy so callers facts aren't changed
    changed = True #loop runs AT LEAST once
    while changed: #loop until nothing else derived
        changed = False #assumes nothing new has been added this pass around
        for conditions, conclusion in rules:
            if conditions <= known and conclusion not in known: #conditions <= known: are all conditions known facts? subset test
                #not in known stops adding the same fact twice
                known.add(conclusion) #rule fires and adds conclusion
                changed = True #loop again if something new opops ip
    return known #starting facts + all derived

#same as above but records trace of fired rules. show why answer was derived
def forward_chain_trace(facts, rules):
    known, trace = set(facts), [] #trace starts as empty list
    changed = True
    while changed:
        changed = False
        for conditions, conclusion in rules:
            if conditions <= known and conclusion not in known:
                known.add(conclusion)
                trace.append((sorted(conditions), conclusion)) #clock why broski
                changed = True
    return known, trace #facts and list of rules fired


row = {"buying": "low", "safety": "high", "persons": "4", "lug_boot": "big"}
#this above is test to show how fwd chain work. dictionary of attributes/values

facts = {f"{attribute}={value}" for attribute, value in row.items()} #attributes=value pair turns into fact )e.g. buying=high

#gna run forward chaining and print just the facts but can derive more than one class. conflict :(
derived = forward_chain(facts, rules) 
print(sorted(f for f in derived if f.startswith("class=")))

#show which rules fired, wat conclusion was
known, trace = forward_chain_trace(facts, rules)
for conditions, conclusion in trace:
    print(f"{' + '.join(conditions)} --> {conclusion}") #trace of fired rules + conclusion. how conclusion was derived

#fwdch derives conclusions from first match conflict res. 
def predict(row, rules, default="class=unacc"): 
    facts = {f"{a}={v}" for a, v in row.items() if a != "class"} 
    derived = forward_chain(facts, rules) #fwd chain derives each possible conclusion
    for conditions, conclusion in rules:  #first priority order
        if conclusion.startswith("class=") and conditions <= derived:
            return conclusion #first match
    return default #nothing derived

#classify will return class value only and not the prefix class=
def classify(r): #this classifies the car based on the relevant rules and returns the relevant class value only
    return predict(r, rules).split("=")[1] #"class=unacc" --> "unacc". comares with to come r["class"]

#predict most common class learned from training (baseline accuracy) and test score
majority = Counter(r["class"] for r in train).most_common(1)[0][0] #how many training cars are of majority class
baseline = sum(r["class"] == majority for r in test) /len(test) #how many test cares are of majority
print(f"Always predict {majority!r}: {baseline:.2f}") #baseline accuracy

    